# LLM-based extraction of past medical history from clinical free text — stroke cohort

Supplementary material for: *<paper title>*.

**Pipeline.** Free-text entries from three sources (past medical history, problem list and the literal reason for the emergency call) are restricted to the stroke cohort, normalised and classified by a locally served large language model (LLM) into a predefined taxonomy (52 categories). A set of deterministic, traceable rules then corrects verifiable errors, and all codes are consolidated into a single multi-label variable per emergency demand, from which a one-hot encoding is derived.

**Availability semantics** (per demand, when no code is extracted): `No_Ant_Interes` = text was available but contained no history of interest; `None` = no text in any source. The category `9_1_Plan_Cronicos` flags inclusion in a chronic-care programme (a care-management marker, not a disease).

**Data availability.** The source data contain identifiable health information and cannot be shared. <data availability statement>. This notebook documents the method; file locations are placeholders and all cell outputs have been removed.

**Computational environment.** Ollama <version>, model `qwen2.5:32b-instruct-q3_K_M` (4-bit-class quantisation), single GPU with 24 GB VRAM, Python <version>, pandas <version>.

## 1. Configuration

In [ ]:
import json
import re
import unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import ollama
import pandas as pd

# Input/output locations (placeholders: the source data are not publicly available)
DATA_DIR = Path('data')
OUT_DIR = Path('output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    'cohort':   DATA_DIR / 'cohort.parquet',    # demand_id, encounter_key
    'history':  DATA_DIR / 'history.parquet',   # encounter_key, text
    'problems': DATA_DIR / 'problems.parquet',  # encounter_key, text
    'reasons':  DATA_DIR / 'reasons.parquet',   # demand_id, text
}
LLM_CACHE = OUT_DIR / 'llm_cache.jsonl'

# LLM configuration: exactly the options sent to the model in the reported run
MODEL = 'qwen2.5:32b-instruct-q3_K_M'
LLM_OPTIONS = {'temperature': 0.1, 'num_ctx': 4096, 'num_predict': 512}
N_WORKERS = 3  # concurrent requests (Ollama server with OLLAMA_NUM_PARALLEL >= N_WORKERS)
RETRIES = 2    # retries on transport errors only

## 2. Input data

Four tables are required. Identifiers are pseudonymised; free text is used as recorded (no spelling correction).

| Table | Grain | Columns |
|---|---|---|
| `cohort` | one row per emergency demand of the stroke cohort | `demand_id`, `encounter_key` (null if the demand has no linked clinical encounter) |
| `history` | one row per past-medical-history entry | `encounter_key`, `text` |
| `problems` | one row per problem-list entry | `encounter_key`, `text` |
| `reasons` | one row per demand (deduplicated beforehand) | `demand_id`, `text` (literal reason for the call) |

History and problem-list entries reach a demand through its encounter; the reason for the call is attached directly to the demand, so demands without a linked encounter are retained. No additional filtering (e.g., by record status or date) is applied.

In [ ]:
def norm_key(s):
    """Normalise a pseudonymised identifier (strip + lower-case), keeping nulls."""
    return s.astype(str).str.strip().str.lower().where(s.notna())

cohort = pd.read_parquet(PATHS['cohort'], columns=['demand_id', 'encounter_key'])
history = pd.read_parquet(PATHS['history'], columns=['encounter_key', 'text'])
problems = pd.read_parquet(PATHS['problems'], columns=['encounter_key', 'text'])
reasons = pd.read_parquet(PATHS['reasons'], columns=['demand_id', 'text'])

for df in (cohort, history, problems):
    df['encounter_key'] = norm_key(df['encounter_key'])
assert reasons['demand_id'].is_unique
print(f'Cohort demands: {cohort["demand_id"].nunique():,}')

## 3. Cohort restriction and stacking of mentions

Each free-text entry ("mention") keeps its demand identifier, which is the final unit of analysis.

In [ ]:
KEY, KEY_DEM = 'encounter_key', 'demand_id'
cohort_keys = set(cohort[KEY].dropna())
enc_to_dem = cohort.dropna(subset=[KEY]).drop_duplicates(KEY).set_index(KEY)[KEY_DEM]

def encounter_mentions(df, source):
    df = df[df[KEY].isin(cohort_keys)]
    return pd.DataFrame({'TEXT': df['text'].astype(str).values, KEY: df[KEY].values,
                         KEY_DEM: df[KEY].map(enc_to_dem).values, 'SOURCE': source})

# Reason for call: attached per demand (demands without a linked encounter are kept)
_r = cohort.drop_duplicates(KEY_DEM)[[KEY_DEM, KEY]].merge(reasons, on=KEY_DEM, how='left')
_r = _r[_r['text'].notna()]
reason_mentions = pd.DataFrame({'TEXT': _r['text'].astype(str).values, KEY: _r[KEY].values,
                                KEY_DEM: _r[KEY_DEM].values, 'SOURCE': 'REASON_FOR_CALL'})

mentions = pd.concat([encounter_mentions(history, 'PAST_HISTORY'),
                      encounter_mentions(problems, 'PROBLEM_LIST'),
                      reason_mentions], ignore_index=True)
assert mentions[KEY_DEM].notna().all()
print(f'Mentions: {len(mentions):,}')
print(mentions['SOURCE'].value_counts().to_string())

## 4. Taxonomy, ICD mapping and prompt

Groups 1–8 follow a taxonomy validated by clinicians; group 9 is a care-management marker; group 10 contains NANDA nursing diagnoses. **The mapping from categories to ICD-10 ranges is an editable proposal and was not part of the clinical validation.** The prompt is reproduced verbatim.

In [ ]:
ICD_MAPPER = {
    # Group 1 - Risk factors
    '1_1_Hypertension': 'I10-I15', '1_2_Diabetes': 'E10-E14', '1_3_Dyslipidemia': 'E78',
    '1_4_Smoking': 'F17/Z72.0', '1_5_Obesity': 'E66', '1_6_Alcohol': 'F10/Z72.1', '1_7_Toxicos': 'F11-F19',
    # Group 2 - Functional / cognitive
    '2_1_Dementia': 'F00-F03', '2_2_Bedridden': 'Z74.0', '2_3_Polimedicado': 'Z91.14',
    # Group 3 - Cardiovascular
    '3_1_Ischemic_Heart': 'I20-I25', '3_2_Heart_Failure': 'I50', '3_3_Arrhythmia': 'I44-I49/Z95.0',
    '3_4_Peripheral_Vasc': 'I70-I73', '3_5_ChronicAortic': 'I71', '3_6_OtherCardiovascular': 'I05-I09/I34-I38',
    '3_7_Sincope': 'R55',
    # Group 4 - Respiratory
    '4_1_COPD': 'J40-J44/J47', '4_2_Asthma': 'J45-J46', '4_3_Oxygen_Support': 'Z99.81/Z99.1',
    '4_4_ETEV': 'I26/I80-I82', '4_5_RespiratoryFailure': 'J96',
    # Group 5 - Neurological
    '5_1_Cerebrovascular': 'I60-I69/G45', '5_2_Epilepsy': 'G40-G41', '5_3_Hemiplegia': 'G80-G83',
    '5_4_NeuroChronic': 'G20-G37/G90-G99',
    # Group 6 - Organ / systemic
    '6_1_Renal': 'N20-N23/Q60-Q64', '6_2_Hepatic': 'B15-B19/K70-K77', '6_3_Oncologic': 'C00-C97',
    '6_4_Connective_Tissue': 'M30-M36', '6_5_Digestive': 'K20-K31/K50-K92', '6_6_RenalChronic': 'N18/Z99.2',
    # Group 7 - Other
    '7_1_OtherQuiru': 'Z98', '7_2_Other_Inmunodepressive': 'B20-B24/D80-D89/Z94', '7_3_Allergy': 'Z88/T78',
    '7_4_OtherInfec': 'A00-B99', '7_5_OtherTrauma': 'M00-M99', '7_Other': '',
    # Group 8 - Mental health
    '8_MentalHealth': 'F20-F99',
    # Group 9 - Care management (no ICD)
    '9_1_Plan_Cronicos': '',
    # Group 10 - NANDA nursing diagnoses (no ICD)
    '10_1_Fall_Risk': '', '10_2_Self_Care_Deficit': '', '10_3_Mobility_Impairment': '', '10_4_Cognitive_Impairment': '',
    '10_5_Activity_Intolerance': '', '10_6_Urinary_Incontinence': '', '10_7_Home_Maintenance': '', '10_8_Therapeutic_Regimen': '',
    '10_9_Sleep_Impairment': '', '10_10_Skin_Integrity_Risk': '', '10_11_Pain_Chronic': '', '10_12_Other_Nursing': '',
}
VALID_CATEGORIES = set(ICD_MAPPER)
assert len(VALID_CATEGORIES) == 52

TAXONOMY_BLOCK = """
TAXONOMY - use ONLY these exact category codes:

=== GRUPO 1: Factores de riesgo ===
  1_1_Hypertension      | HTA, hipertension, tension alta, hipertenso. NOT: HBP (prostata).
  1_2_Diabetes          | DM, DM2, diabetes, insulinodependiente, pie diabetico.
  1_3_Dyslipidemia      | DLP, dislipemia, colesterol, trigliceridos, hiperlipemia.
  1_4_Smoking           | tabaquismo, fumador, exfumador. ONLY tobacco (NOT cannabis).
  1_5_Obesity           | obesidad, sobrepeso, IMC elevado.
  1_6_Alcohol           | alcohol, OH, enolismo, bebedor, ex-bebedor. ONLY alcohol.
  1_7_Toxicos           | drogas, toxicos, cocaina, heroina, cannabis, marihuana, adiccion (excluido alcohol y tabaco).

=== GRUPO 2: Funcional / cognitivo ===
  2_1_Dementia          | demencia, Alzheimer, deterioro cognitivo, senil.
  2_2_Bedridden         | encamado, postrado, inmovilidad, dependencia funcional, vida cama-sillon.
  2_3_Polimedicado      | polimedicado, polifarmacia, multiples farmacos cronicos.

=== GRUPO 3: Cardiovascular ===
  3_1_Ischemic_Heart    | cardiopatia isquemica, IAM, angina, stent, bypass, revascularizacion, SCA, infarto miocardio.
  3_2_Heart_Failure     | insuficiencia cardiaca, IC, ICC, EAP, edema agudo pulmon.
  3_3_Arrhythmia        | FA, ACxFA, fibrilacion auricular, flutter, arritmia, marcapasos, DAI, bloqueo AV.
  3_4_Peripheral_Vasc   | arteriopatia periferica, isquemia arterial de miembros, claudicacion intermitente. SOLO arterial. NOT varices/insuf venosa (-> 7_Other), NOT flebitis/TVP/TEP (-> 4_4_ETEV).
  3_5_ChronicAortic     | aneurisma aortico, dilatacion de aorta. NOT estenosis/insuficiencia aortica valvular (-> 3_6_OtherCardiovascular).
  3_6_OtherCardiovascular| valvulopatia, estenosis/insuficiencia aortica (valvular), soplo, HTP, miocardiopatia, pericarditis, otra cardiopatia.
  3_7_Sincope           | sincope, lipotimia, presincope, perdida de conocimiento vasovagal.

=== GRUPO 4: Respiratorio ===
  4_1_COPD              | EPOC, enfisema, bronquitis cronica, bronquiectasias.
  4_2_Asthma            | asma, hiperreactividad bronquial.
  4_3_Oxygen_Support    | oxigenoterapia, O2 domiciliario, CPAP, VMNI. NOT SAOS alone.
  4_4_ETEV              | TEP, tromboembolismo pulmonar, TVP (aguda o cronica), flebitis, tromboflebitis, enfermedad tromboembolica venosa.
  4_5_RespiratoryFailure| insuficiencia respiratoria, IRA, IRC respiratoria, hipoxemia cronica.

=== GRUPO 5: Neurológico ===
  5_1_Cerebrovascular   | ictus, ACV, ACVA, AIT, infarto cerebral, accidente vascular. NOT FA alone.
  5_2_Epilepsy          | epilepsia, crisis convulsivas, comicialidad.
  5_3_Hemiplegia        | hemiplejia, paraplejia, tetraplejia, paresia PERMANENTE. NOT artrosis/escoliosis.
  5_4_NeuroChronic      | Parkinson, esclerosis multiple, ELA, neuropatia, otra enf. neurologica cronica.

=== GRUPO 6: Órgano / sistémico ===
  6_1_Renal             | litiasis renal, nefrolitiasis, colico renal, patologia renal estructural SIN insuficiencia.
  6_2_Hepatic           | cirrosis, hepatopatia, insuficiencia hepatica, hepatitis cronica.
  6_3_Oncologic         | cancer, neoplasia, tumor maligno, leucemia, linfoma, metastasis, carcinoma.
  6_4_Connective_Tissue | lupus, artritis reumatoide, vasculitis, Sjogren, enf. autoinmune.
  6_5_Digestive         | hernia de hiato, ulcera gastrica, reflujo, colitis, EII, patologia digestiva.
  6_6_RenalChronic      | IRC, ERC, insuficiencia renal cronica, dialisis, trasplante renal.

=== GRUPO 7: Otros ===
  7_1_OtherQuiru        | antecedente quirurgico, cualquier cirugia previa, intervencion.
  7_2_Other_Inmunodepressive| VIH, esplenectomia, trasplante, inmunodepresion, tratamiento inmunosupresor.
  7_3_Allergy           | alergia, alergico a, hipersensibilidad, alergia medicamentosa/alimentaria.
  7_4_OtherInfec        | antecedente infeccioso, TBC, infeccion cronica, hepatitis viral resuelta.
  7_5_OtherTrauma       | artrosis, gonartrosis, espondiloartrosis, osteoporosis, gota, hiperuricemia, fractura, patologia musculoesqueletica.
  7_Other               | otra patologia clinica sin grupo especifico (hipotiroidismo, glaucoma, SAOS, fibromialgia).

=== GRUPO 8: Salud mental ===
  8_MentalHealth        | depresion, ansiedad, trastorno bipolar, esquizofrenia, psiquiatria, trastorno salud mental.

=== GRUPO 9: Gestión asistencial / programas ===
  9_1_Plan_Cronicos      | Inclusion en programa de cronicos: "plan de cronicos", "plan de cronicos complejo", "paciente cronico complejo", "PCC", "PAP cronicos", "programa de cronicos". Es un MARCADOR DE GESTION (pluripatologia/fragilidad), NO una patologia. Si el texto ademas menciona patologias, devolver 9_1_Plan_Cronicos ADEMAS del codigo clinico correspondiente.

=== GRUPO 10: Diagnósticos de enfermería NANDA ===
  10_1_Fall_Risk         | Riesgo de caidas, riesgo de caida.
  10_2_Self_Care_Deficit | Deficit de autocuidado: baño, higiene, vestido, alimentacion, uso del WC.
  10_3_Mobility_Impairment| Deterioro de la movilidad fisica, deterioro deambulacion, riesgo sindrome desuso.
  10_4_Cognitive_Impairment| Deterioro de la memoria, confusion cronica, trastorno procesos pensamiento.
  10_5_Activity_Intolerance| Intolerancia a la actividad, fatiga, riesgo intolerancia actividad.
  10_6_Urinary_Incontinence| Incontinencia urinaria funcional / de urgencia / total.
  10_7_Home_Maintenance  | Deterioro mantenimiento del hogar, alteracion mantenimiento de la salud.
  10_8_Therapeutic_Regimen| Manejo (in)efectivo del regimen terapeutico, conductas generadoras de salud.
  10_9_Sleep_Impairment  | Deterioro del patron del sueño, insomnio, trastorno del sueño.
  10_10_Skin_Integrity_Risk| Riesgo/deterioro integridad cutanea, ulcera por presion.
  10_11_Pain_Chronic     | Dolor cronico, dolor agudo.
  10_12_Other_Nursing    | cualquier otro diagnostico enfermero NANDA no clasificable arriba.

STRICT RULES:
1. ONLY return codes from the list above. NEVER invent codes.
2. Use the FULL code name (e.g. "3_3_Arrhythmia", NOT "3_3").
3. NEGATIONS ("No HTA", "Niega", "Sin antecedentes") -> return [].
4. NOISE (symbols, numbers only, "ver informe", "Las descritas") -> return [].
5. NO DUPLICATES in the list.
6. Medical/clinical findings -> groups 1-8. Nursing (NANDA) diagnoses -> group 10. Never mix them up.
7. alcohol/OH/enolismo -> 1_6_Alcohol (NOT 1_7). cannabis/drogas -> 1_7_Toxicos.
8. litiasis/nefrolitiasis SIN insuficiencia -> 6_1_Renal. IRC/ERC/dialisis -> 6_6_RenalChronic.
9. artrosis, gota, osteoporosis, hiperuricemia -> 7_5_OtherTrauma (NOT 5_3, NOT None).
10. depresion/ansiedad/psiquiatria -> 8_MentalHealth.
11. "plan de cronicos"/"paciente cronico complejo"/PCC/PAP cronicos -> 9_1_Plan_Cronicos (marcador de gestion; NO confundir con "X cronica" que es adjetivo de una patologia, p.ej. "bronquitis cronica" -> 4_1_COPD, NOT 9_1).
12. ACUTE-EVENT FILTER (critical). This text may be a CURRENT reason for consultation, not past history. If the text describes an ACUTE EPISODE or CURRENT SYMPTOMS of the patient (verbs in present/recent past narrating what is happening now: "no puede mover", "se ha caido", "esta desorientado", "perdida de fuerza", "no habla", "mareo", "cefalea", "disartria", "afasia", "vomitos", "dolor toracico ahora"), that description is NOT an antecedent -> do NOT emit a code for it. It is handled by a separate process.
13. CHRONIC EXCEPTION. Within an acute text, DO extract a condition ONLY if it is a chronic disease or risk factor named as established history (e.g. "HTA", "diabetes", "EPOC", "antecedentes de ictus", "conocido X", "en tratamiento cronico por X"). Example: "crisis de hta con disartria transitoria" -> ["1_1_Hypertension"] (HTA is chronic history; disartria is the acute symptom, dropped).
14. Do NOT infer chronic disease from an acute symptom. "no puede mover el brazo" is NOT 5_1_Cerebrovascular; "se ha caido" is NOT 10_1_Fall_Risk; "desorientado" is NOT 2_1_Dementia nor 10_4. Only code these if named as history ("antecedente de ictus" -> 5_1).
15. NANDA (group 10) codes apply ONLY to an ESTABLISHED nursing diagnosis or a chronic/recurrent condition ("riesgo de caidas" as a documented diagnosis, "dolor cronico", "incontinencia urinaria cronica"), NEVER to a one-off acute symptom described in the text ("dolor de cabeza hoy", "se cayo esta manana").
16. Cardiovascular boundaries: varices / insuficiencia venosa (without thrombosis) -> 7_Other. flebitis / TVP / TEP -> 4_4_ETEV. estenosis/insuficiencia aortica (valvular) -> 3_6_OtherCardiovascular. aneurisma/dilatacion aortica -> 3_5_ChronicAortic.

EXAMPLES:
  "HTA, FA" -> ["1_1_Hypertension", "3_3_Arrhythmia"]
  "EPOC ENFISEMA PULMONAR" -> ["4_1_COPD"]
  "Deja de beber OH" -> ["1_6_Alcohol"]
  "CONSUMO DE COCAINA" -> ["1_7_Toxicos"]
  "NEFROLITIASIS" -> ["6_1_Renal"]
  "ERC EN DIALISIS" -> ["6_6_RenalChronic"]
  "GONARTROSIS ESCOLIOSIS" -> ["7_5_OtherTrauma"]
  "DEPRESION MAYOR" -> ["8_MentalHealth"]
  "Riesgo de caidas" -> ["10_1_Fall_Risk"]
  "Deterioro de la movilidad fisica" -> ["10_3_Mobility_Impairment"]
  "COLECISTECTOMIA" -> ["7_1_OtherQuiru"]
  "plan de cronicos" -> ["9_1_Plan_Cronicos"]
  "plan cronicos ic" -> ["9_1_Plan_Cronicos", "3_2_Heart_Failure"]
  "paciente cronico complejo" -> ["9_1_Plan_Cronicos"]
  "No AMC" -> []
  "****" -> []
  "varon sincope" -> []
  "se ha caido hoy tres veces" -> []
  "no puede mover el brazo" -> []
  "crisis de hta con disartria transitoria" -> ["1_1_Hypertension"]
  "su marido con antecedentes de ictus con brazo inmovilizado" -> ["5_1_Cerebrovascular"]
  "acude por disnea, hta y diabetes conocidas" -> ["1_1_Hypertension", "1_2_Diabetes"]
  "sind varicoso en mii" -> ["7_Other"]
  "insuficiencia venosa mmii" -> ["7_Other"]
  "flebitis en 2016" -> ["4_4_ETEV"]
  "estenosis aortica severa" -> ["3_6_OtherCardiovascular"]
"""

SYSTEM_PROMPT = (
    "You are a clinical coding expert. Classify Spanish medical records using ONLY the taxonomy below. "
    "Return ONLY valid JSON. No explanations.\n"
    + TAXONOMY_BLOCK
)
USER_TEMPLATE = 'Classify: "%s"\nReturn JSON: {"predicted_categories": ["code1", "code2"]}'

# Structured output: the model can only emit taxonomy codes
OUTPUT_SCHEMA = {
    'type': 'object',
    'properties': {'predicted_categories': {
        'type': 'array', 'uniqueItems': True,
        'items': {'type': 'string', 'enum': sorted(VALID_CATEGORIES)}}},
    'required': ['predicted_categories'],
    'additionalProperties': False,
}

## 5. LLM classification

Texts are normalised (accents removed, lower-case, whitespace collapsed; punctuation kept because it separates findings). Each distinct normalised text is classified once, one text per request. Codes outside the taxonomy are discarded; a text with no code is labelled `No_Ant_Interes`. Texts whose request fails are left unclassified (`None`). Results are cached so the run can be resumed.

*Note on determinism:* with `temperature = 0.1` and concurrent requests, repeated runs are not guaranteed to be bitwise identical.

In [ ]:
def normalize_literal(text):
    s = str(text).strip().lower()
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', s).strip()

def classify(text):
    """Raw (pre-remediation) label for one normalised text, or None if the request fails."""
    for _ in range(RETRIES + 1):
        try:
            resp = ollama.chat(model=MODEL, format=OUTPUT_SCHEMA, options=LLM_OPTIONS,
                               messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
                                         {'role': 'user', 'content': USER_TEMPLATE % text}])
        except Exception:  # transport error: retry
            continue
        try:
            cats = json.loads(resp['message']['content'])['predicted_categories']
        except (ValueError, KeyError, TypeError):  # malformed output: not retried
            return None
        cats = list(dict.fromkeys(c for c in cats if c in VALID_CATEGORIES))
        return ', '.join(cats) if cats else 'No_Ant_Interes'
    return None

mentions['TEXT_NORM'] = mentions['TEXT'].map(normalize_literal)
unique_texts = list(mentions.loc[mentions['TEXT_NORM'].str.len() > 0, 'TEXT_NORM']
                    .value_counts().index)  # most frequent first

cache = {}
if LLM_CACHE.exists():
    with open(LLM_CACHE, encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            cache[d['text']] = d['label']
pending = [t for t in unique_texts if t not in cache]
print(f'Unique texts: {len(unique_texts):,} | cached: {len(cache):,} | pending: {len(pending):,}')

with open(LLM_CACHE, 'a', encoding='utf-8') as f, ThreadPoolExecutor(N_WORKERS) as ex:
    for text, label in zip(pending, ex.map(classify, pending)):
        if label is not None:
            cache[text] = label
            f.write(json.dumps({'text': text, 'label': label}, ensure_ascii=False) + '\n')

mentions['CATEGORY_LLM'] = mentions['TEXT_NORM'].map(cache).fillna('None')
print(f'Unresolved texts: {sum(t not in cache for t in unique_texts):,}')

## 6. Deterministic remediation

Rules applied to the raw text and the LLM label. They do not reclassify; they correct errors verifiable against the taxonomy:

- **A** keep only canonical taxonomy codes;
- **B** substance terms → `1_6_Alcohol` / `1_7_Toxicos`; benign prostatic hyperplasia → `7_Other` (not hypertension);
- **C** oncological terms missing from the prompt's synonym list (melanoma, adenocarcinoma, sarcoma, glioma, myeloma, blastoma, "Ca de <organ>") → `6_3_Oncologic`, unless negated; prostate cancer is not additionally labelled `7_Other`;
- **D** cannabis without tobacco is not `1_4_Smoking` but `1_7_Toxicos`;
- **E** chronic-care programme mentions → `9_1_Plan_Cronicos` (additive);
- **F** varices/venous insufficiency without thrombosis → `7_Other`; phlebitis/DVT/PE → `4_4_ETEV` (never `3_4`, which is arterial).

Rule tags are recorded in `TRACE`.

In [ ]:
I = re.IGNORECASE
SUBSTANCES_RE = re.compile(r'\b(enolismo|bebedor|bebedora|alcohol|cerveza|coca[íi]na|hero[íi]na|'
                           r'cannabis|drogas|hbp|pr[óo]stata)\b', I)
ALCOHOL_RE = re.compile(r'\b(enolismo|bebedor|bebedora|alcohol|cerveza)\b', I)
DRUGS_RE = re.compile(r'\b(coca[íi]na|hero[íi]na|cannabis|drogas)\b', I)
ANY_SUBSTANCE_RE = re.compile(r'\b(enolismo|bebedor|bebedora|alcohol|cerveza|coca[íi]na|hero[íi]na|cannabis|drogas)\b', I)
NEG_RE = re.compile(r'\b(no|niega|niego|niegan|sin|descart\w*|nunca|ausencia de)\b', I)
ONCOLOGY_RE = re.compile(
    r'\b(melanoma|adenocarcinoma|sarcoma|glioma|mieloma|blastoma)\b|'
    r'\bca\.?\s+(de\s+)?(mama|colon|pr[óo]stata|pulm[óo]n|vejiga|vesical|recto|'
    r'est[óo]mago|p[áa]ncreas|ri[nñ][óo]n|ovario|[úu]tero|endometrio|es[óo]fago|'
    r'laringe|tiroides|piel|h[íi]gado|sigma|cervix)\b', I)
PROSTATE_CA_RE = re.compile(r'\bca\.?\s+(de\s+)?pr[óo]stata\b|adenocarcinoma\s+(de\s+)?pr[óo]stata', I)
HBP_RE = re.compile(r'\bhbp\b|hiperplasia\s+benigna', I)
CANNABIS_RE = re.compile(r'\b(cannabis|marihuana|marij\w*|porros?|hach[íi]s|THC)\b', I)
TOBACCO_RE = re.compile(r'\b(tabaco|tabaquismo|cigarr\w+|puros?|pipa|fumador de tabaco)\b', I)
CHRONIC_PLAN_RE = re.compile(r'\bplan\b[^.]{0,20}\bcronic|\bpaciente\s+cronico|\bpcc\b|'
                             r'\bpap\b[^.]{0,20}cronic|programa[^.]{0,20}cronic', I)
VARICES_RE = re.compile(r'\bvarices?\b|\bvaricos\w*\b|insuf\w*\s+venos\w+|\bivc\b', I)
THROMBO_RE = re.compile(r'\bflebitis\b|tromboflebitis|\btvp\b|\btep\b|trombosis\s+venos\w+|tromboembol\w+', I)

def clean_row_category(text, llm_label):
    """Returns (remediated label, trace)."""
    text, llm_label = str(text), str(llm_label)
    low = text.lower()
    trace = []
    # (A) canonical codes only (availability labels are not codes)
    final = {c.strip() for c in llm_label.split(',') if c.strip() in VALID_CATEGORIES}
    # (B) substances and benign prostatic hyperplasia
    if SUBSTANCES_RE.search(text):
        if ALCOHOL_RE.search(text):
            final.add('1_6_Alcohol')
        if DRUGS_RE.search(text):
            final.add('1_7_Toxicos')
        if 'hbp' in low or 'próstata' in low or 'prostata' in low:
            final.add('7_Other')
            final.discard('1_1_Hypertension')
    # (C) oncology terms missing from the prompt
    if ONCOLOGY_RE.search(text) and not NEG_RE.search(text):
        if '6_3_Oncologic' not in final:
            final.add('6_3_Oncologic')
            trace.append('onco_texto')
        if PROSTATE_CA_RE.search(text) and not HBP_RE.search(text) and not ANY_SUBSTANCE_RE.search(text):
            final.discard('7_Other')
            trace.append('prostata_cancer_no_7other')
    # (D) cannabis is not tobacco
    if CANNABIS_RE.search(text) and not TOBACCO_RE.search(text):
        if '1_4_Smoking' in final:
            final.discard('1_4_Smoking')
            trace.append('cannabis_no_tabaco')
        final.add('1_7_Toxicos')
    # (E) chronic-care programme marker (additive)
    if CHRONIC_PLAN_RE.search(text) and '9_1_Plan_Cronicos' not in final:
        final.add('9_1_Plan_Cronicos')
        trace.append('plan_cronicos')
    # (F) venous vs arterial boundary
    varices, thrombo = VARICES_RE.search(text), THROMBO_RE.search(text)
    if varices and not thrombo:
        if '3_4_Peripheral_Vasc' in final:
            final.discard('3_4_Peripheral_Vasc')
            trace.append('varices_no_3_4')
        final.add('7_Other')
    if thrombo and not NEG_RE.search(text):
        if '3_4_Peripheral_Vasc' in final:
            final.discard('3_4_Peripheral_Vasc')
            trace.append('flebitis_4_4')
        final.add('4_4_ETEV')
    # keep the availability label when no code remains
    if final:
        label = ', '.join(sorted(final))
    elif llm_label.strip().lower() == 'no_ant_interes':
        label = 'No_Ant_Interes'
    else:
        label = 'None'
    return label, '; '.join(trace)

_fix = [clean_row_category(t, c) for t, c in zip(mentions['TEXT'], mentions['CATEGORY_LLM'])]
mentions['CATEGORY'] = [x[0] for x in _fix]
mentions['TRACE'] = [x[1] for x in _fix]

_changed = sum(set(a.split(', ')) != set(b.split(', '))
               for a, b in zip(mentions['CATEGORY_LLM'], mentions['CATEGORY']))
print(f'Mentions whose label changed after remediation: {_changed:,} of {len(mentions):,}')
print('Traced rules:', dict(Counter(r for t in mentions['TRACE'] if t for r in t.split('; '))))

## 7. Consolidation per demand

All codes from the three sources are merged and deduplicated per demand. Demands with no code are labelled `No_Ant_Interes` if at least one mention had text, and `None` otherwise.

In [ ]:
mentions['HAS_TEXT'] = mentions['CATEGORY'].str.strip().str.lower().ne('none')

def clinical_codes(series):
    return sorted({t.strip() for cell in series.dropna() for t in str(cell).split(',')
                   if t.strip() in VALID_CATEGORIES})

demands = pd.Index(cohort[KEY_DEM].dropna().unique(), name=KEY_DEM)
_codes = mentions.groupby(KEY_DEM)['CATEGORY'].apply(clinical_codes).reindex(demands)
_text = mentions.groupby(KEY_DEM)['HAS_TEXT'].any().reindex(demands, fill_value=False)

per_demand = pd.DataFrame({KEY_DEM: demands,
                           'codes': [c if isinstance(c, list) else [] for c in _codes],
                           'tiene_texto': _text.astype(int).values})
per_demand['ANTECEDENTES_UNIFICADOS'] = [
    ', '.join(c) if c else ('No_Ant_Interes' if t else 'None')
    for c, t in zip(per_demand['codes'], per_demand['tiene_texto'])]
per_demand['ICD_UNIFICADO'] = [
    ', '.join(sorted({ICD_MAPPER[x] for x in c if ICD_MAPPER[x]})) or 'None'
    for c in per_demand['codes']]
per_demand['N_ANTECEDENTES'] = per_demand['codes'].str.len()

_disp = per_demand['ANTECEDENTES_UNIFICADOS']
print(f'Demands: {len(per_demand):,}')
print(f'  with >=1 code:   {(per_demand["N_ANTECEDENTES"] > 0).sum():,}')
print(f'  No_Ant_Interes:  {_disp.eq("No_Ant_Interes").sum():,}')
print(f'  None:            {_disp.eq("None").sum():,}')

## 8. One-hot encoding

Availability labels are excluded; availability is carried by `tiene_texto`.

In [ ]:
ohe = per_demand['ANTECEDENTES_UNIFICADOS'].str.get_dummies(sep=', ')
ohe = ohe.drop(columns=[c for c in ohe.columns if c.lower() in {'none', 'no_ant_interes'}])
ohe.columns = ['ant_' + c.lower() for c in ohe.columns]

features = pd.concat([per_demand[[KEY_DEM, 'ANTECEDENTES_UNIFICADOS', 'ICD_UNIFICADO',
                                  'N_ANTECEDENTES', 'tiene_texto']], ohe], axis=1)
print(f'{ohe.shape[1]} indicator columns over {len(features):,} demands')

## 9. Integrity checks

These verify structural consistency, not clinical accuracy.

In [ ]:
disp = per_demand['ANTECEDENTES_UNIFICADOS']
n_code = int((per_demand['N_ANTECEDENTES'] > 0).sum())
ant_cols = [c for c in features.columns if c.startswith('ant_')]
checks = {
    'demand_id unique': per_demand[KEY_DEM].is_unique,
    'availability partition is exhaustive':
        n_code + disp.eq('No_Ant_Interes').sum() + disp.eq('None').sum() == len(per_demand),
    'No_Ant_Interes implies text': (per_demand.loc[disp.eq('No_Ant_Interes'), 'tiene_texto'] == 1).all(),
    'None implies no text': (per_demand.loc[disp.eq('None'), 'tiene_texto'] == 0).all(),
    'one-hot row sums equal N_ANTECEDENTES':
        (features[ant_cols].sum(axis=1) == features['N_ANTECEDENTES']).all(),
    'one-hot columns within taxonomy':
        set(ant_cols) <= {'ant_' + c.lower() for c in VALID_CATEGORIES},
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)
assert all(checks.values())

## 10. Output

In [ ]:
per_demand.drop(columns='codes').to_parquet(OUT_DIR / 'history_per_demand.parquet', index=False)
features.to_parquet(OUT_DIR / 'history_one_hot.parquet', index=False)